In [1]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR

PosixPath('/workspaces/GPRtest/data')

## 1. 実行環境の準備

- プロジェクトルートと `src/` のパスを設定します。
- `fx_data_fetcher.py` をimportできるように `sys.path` を調整します。
- 出力先の `data/` ディレクトリを作成（なければ自動作成）します。

# FXデータ取得ノートブック

> このノートブックでは、為替データを取得して `data/` 配下にCSV保存します。

- 対応ソース: `stooq` / `fred`
- 対応通貨ペア: 例 `USDJPY`, `EURUSD`
- 保存先: `data/<pair>.csv`

In [2]:
import pandas as pd
from fx_data_fetcher import FXDataFetcher

# Parameters
source = "stooq"  # options: 'stooq' or 'fred'
start_date = "2010-01-01"  # set None for full history
end_date = None
fred_api_key = os.getenv("FRED_API_KEY") if source == "fred" else None

fetcher = FXDataFetcher(
    source=source,
    data_dir=DATA_DIR,
    fred_api_key=fred_api_key,
)
fetcher

FXDataFetcher(source='stooq', data_dir=PosixPath('/workspaces/GPRtest/data'), fred_api_key=None, timeout=30)

## 2. 取得条件とフェッチャー初期化

- データ取得元（`source`）と期間（`start_date`, `end_date`）を設定します。
- `fred` を使う場合は環境変数 `FRED_API_KEY` が必要です。
- `FXDataFetcher` を初期化して、以降の取得処理で使います。

In [3]:
pairs = ["USDJPY", "EURUSD"]
dfs = {}
output_paths = {}

for pair in pairs:
    df_pair = fetcher.fetch_pair(pair, start_date=start_date, end_date=end_date)
    output_path = DATA_DIR / f"{pair.lower()}.csv"
    fetcher.save_csv(df_pair, output_path)
    dfs[pair] = df_pair
    output_paths[pair] = output_path

dfs.keys(), output_paths

(dict_keys(['USDJPY', 'EURUSD']),
 {'USDJPY': PosixPath('/workspaces/GPRtest/data/usdjpy.csv'),
  'EURUSD': PosixPath('/workspaces/GPRtest/data/eurusd.csv')})

## 3. 通貨ペアごとのデータ取得と保存

- `pairs` に指定した通貨ペアをループ処理します。
- 各ペアの時系列データを取得し、`data/<pair>.csv` として保存します。
- 取得済みデータフレームと保存パスを辞書に保持します。

In [4]:
for pair, df_pair in dfs.items():
    print(f"{pair} rows: {len(df_pair)}")
    display(df_pair.head())
    print(f"Saved: {output_paths[pair]}")

USDJPY rows: 4174


,date,close
0,2010-01-04,92.40
1,2010-01-05,91.72
2,2010-01-06,92.34
3,2010-01-07,93.59
4,2010-01-08,92.65


Saved: /workspaces/GPRtest/data/usdjpy.csv
EURUSD rows: 4174


,date,close
0,2010-01-04,1.4412
1,2010-01-05,1.4361
2,2010-01-06,1.4399
3,2010-01-07,1.4304
4,2010-01-08,1.4398


Saved: /workspaces/GPRtest/data/eurusd.csv


## 4. 取得結果の確認

- ペアごとの件数、先頭行、保存先を表示して内容を確認します。
- 必要に応じて `pairs` や期間設定を変更し、再実行してください。